# TP05 - Attribute Grammars and LarkAG

## Grammar Engineering

## MEI/2025-26

#### Nuno Macedo
Universidade do Minho

The following exercises are intended to be solved with pen and paper and implemented with an AG processor for Python, [LarkAG](https://larkag.epl.di.uminho.pt).

# Processing Attribute Grammars

- The semantics defined by an attribute grammar (AG) can be implemented manually using syntax-directed translations or traversals of abstract syntax trees (ASTs)
  - Cumbersome and error-prone

- However, it is also possible to process them directly and generate a translator that applies the attribute evaluation rules

- Allows us to just specify a context-free grammar (CFG), the attributes associated with each syntactic variable, and their evaluation rules

- The order of traversal and attribute computation is then determined by the AG processor

## AGs in Python and `Lark`

- `Lark` does not support AGs directly

- `LarkAG` is a (prototype) extension that supports AGs on top of `Lark` grammars, and translates them into `Lark` AST traversals

- It uses the `Lark` `Interperter` class, which allows full control on the order of traversal, essential to process syntesized and inherited attributes

- `Lark` `Intereperters` traverse the tree bottom-down and give full control of the traversal

- This is a prototype tool with some limitations, but highlights the main strengths of thinking at the AG level:
  - Decoupling semantics from the parser and implementation details

- Note: the `LarkAG` version in PyPI is not up-to-date, so you should manually install the latest version:
  - Download the source code from the [repository](https://github.com/danielfaria089/LarkAG)
  - Run locally `pip install ./LarkAG`

In [ ]:
pip install lark-ag

## AGs in `LarkAG`

- Start by defining the attributes of each syntactic variable, either synthesized (`SA`) or inherited (`IA`)

- Write a CFG using `Lark` notation, but each production can be interleaved with 3 sets of rules:
  - evaluation rules `ER`, declaritevly determing the value of the attributes from other attributes
  - context conditions `CC`, determine necessary conditions for evaluation, fail otherwise
  - translation rules `TR`, further transformations to be triggered beyond attribute evaluation, arbitrary code

- When the same non-terminal occurs more than once in a production rule, use indexes to select

## Example: Evaluate arithmetic expressions with declared variables

Let's recall the running example of arithmetic expressions with declared variables, and its encoding as an AG

- `A(start) = { (val, synthesized) }`
- `A(decls) = { (val, synthesized) }`
- `A(decl) = { (val, synthesized), (env, inherited) }`
- `A(expr) = { (val, synthesized), (env, inherited) }`
- `A(term) = { (val, synthesized), (env, inherited) }`
- `A(factor) = { (val, synthesized), (env, inherited) }`
- `A(literal) = { (val, synthesized), (env, inherited) }`

| production (R)         | attributes evaluation (ER)                  |
|------------------------|---------------------------------------------|
|`start : decls E `      | `expr.env = decls.val`                            |
|                        | `start.val = expr.val`                         |
|`decls₀ : decls₁ decl`  | `decl.env = decls₁.val`                           |
|                        | `decls₀.val = decl.val`                           |
|`decls : ε`             | `decls.val = {}`                               |
|`decl  : let ID = expr` | `expr.env = decl.env`                             |
|                        | `decl.val = extend(decl.env, ID.lexval → expr.val)`  |
|`expr₀ : expr₁ + term`  | `expr₁.env = expr₀.env`                           |
|                        | `T.env = expr₀.env`                            |
|                        | `expr₀.val = expr₁.val + T.val`                   |
|`expr : term`           | `term.env = expr.env`                             |
|                        | `expr.val = term.val`                             |
|`term₀ : term₁ * factor`| `term₁.env = term₀.env`                           |
|                        | `factor.env = Tterm.env`                            |
|                        | `term₀.val = term₁.val + factor.val`                   |
|`term : factor`         | `factor.env = term.env`                             |
|                        | `term.val = factor.val`                             |
|`factor : -factor`      | `factor₁.env = factor₀.env`                           |
|                        | `factor₀.val = factor₁`.                              |
|`factor : literal`      | `literal.env = factor.env`                             |
|                        | `factor.val = literal.val`                             |
|`literal : ( expr )`    | `expr.env = literal.env`                             |
|                        | `literal.val = expr.val`                             |
|`literal : ID`          | `literal.val = literal.env(ID.lexval)`                  |
|`literal : INT`         | `literal.val = INT.lexval`                        |


In [ ]:
grammar = r"""
SA(start) { val: float }
SA(expr) { val: float }
IA(expr) { env: dict }

SA(term) { val: float }
IA(term) { env: dict }

SA(factor) { val: float }
IA(factor) { env: dict }

SA(literal) { val: float }
IA(literal) { env: dict }

SA(decls) { val: dict }
SA(decl)  { val: dict }
IA(decl)  { env: dict }

start :
decls expr
ER	{
  start.val = expr.val;
  expr.env = decls.val;
}

decls :
decls decl
ER	{
  decls[1].val = decl.val;
  decl.env = decls[2].val;
}
|
ER	{
  decls.val = {};
}

decl :
"let" ID "=" expr ";"
ER	{
  decl.val = decl.env | { str(ID): expr.val };
  expr.env = decl.env;
}

expr :
expr ADD term
ER	{
  expr[1].val = expr[2].val + term.val;
  expr[2].env = expr[1].env;
  term.env = expr[1].env;
}
| expr SUB term
ER	{
  expr[1].val = expr[2].val - term.val;
  expr[2].env = expr[1].env;
  term.env = expr.env;
}
| term
ER	{
  expr.val = term.val;
  term.env = expr.env;
}

term :
term MUL factor
ER	{
  term[1].val = term[2].val * factor.val;
  term[2].env = term[1].env;
  factor.env = term[1].env;
}
| term DIV factor
ER	{
  term[1].val = term[2].val / factor.val;
  term[2].env = term[1].env;
  factor.env = term[1].env;
}
| factor
ER	{
  term.val = factor.val;
  factor.env = term.env;
}

factor :
SUB factor
ER	{
  factor[1].val = - factor[2].val;
  factor[2].env = factor[1].env;
}
| literal
ER	{
  factor.val = literal.val;
  literal.env = factor.env;
}

literal :
ID
ER	{
  literal.val = literal.env.get(str(ID),0);
}
CC {
    str(ID) in literal.env -> "Unbound variable";
}

| NUMBER
ER	{
  literal.val = float(NUMBER);
}
| "(" expr ")"
ER	{
  literal.val = expr.val;
  expr.env = literal.env;
}

ADD: "+"
SUB: "-"
MUL: "*"
DIV: "/"

ID : /[a-z]+/
NUMBER: /[0-9]+(\.[0-9]+)?/
"""

In [ ]:
parser = lark_ag.Lark_AG(grammar)

In [ ]:
parser.process("let x = 20; let y = x*2; x + -y").val

In [ ]:
parser.process("let x = 20; let y = x*2; x + -y + z").val

## `LarkAG` behind the curtain

In [ ]:
parser.getCFG()

In [ ]:
grammar = r"""
start : decls expr -> start_0
decls : decls decl -> decls_0 |  -> decls_1
decl : "let" ID "=" expr ";" -> decl_0
expr : expr ADD term -> expr_0 | expr SUB term -> expr_1 | term -> expr_2
term : term MUL factor -> term_0 | term DIV factor -> term_1 | factor -> term_2
factor : SUB factor -> factor_0 | literal -> factor_1
literal : ID -> literal_0 | NUMBER -> literal_1 | "(" expr ")" -> literal_2
ADD : "+"
SUB : "-"
MUL : "*"
DIV : "/"
ID : /[a-z]+/
NUMBER : /[0-9]+(\.[0-9]+)?/

%import common.WS
%ignore WS
"""

In [ ]:
parser.getInterpreterFile()

In [ ]:
from lark import Lark, Token, Tree
from lark.visitors import Interpreter

class MyInterpreter(Interpreter):
    def __helper__(self, node):
        pointers={}
        pointers[node.data.split('_')[0]] = [node]
        for child in node.children:
            name = ''
            if type(child) == Tree:
                name = child.data.split('_')[0]
            elif type(child) == Token:
                name = child.type
            if name not in pointers:
                pointers[name] = []
            pointers[name].append(child)
        return pointers

    def start_0(self, node):
        pointers = self.__helper__(node)
        self.visit(pointers['decls'][0])
        pointers['expr'][0].env = pointers['decls'][0].val
        self.visit(pointers['expr'][0])
        node.val = pointers['expr'][0].val

    def decls_0(self, node):
        pointers = self.__helper__(node)
        self.visit(pointers['decls'][1])
        pointers['decl'][0].env = pointers['decls'][1].val
        self.visit(pointers['decl'][0])
        node.val = pointers['decl'][0].val

    def decls_1(self, node):
        pointers = self.__helper__(node)
        node.val = {}

    def decl_0(self, node):
        pointers = self.__helper__(node)
        pointers['expr'][0].env = node.env
        self.visit(pointers['expr'][0])
        node.val = node.env | { str(pointers['ID'][0]): pointers['expr'][0].val }

    def expr_0(self, node):
        pointers = self.__helper__(node)
        pointers['term'][0].env = node.env
        self.visit(pointers['term'][0])
        pointers['expr'][1].env = node.env
        self.visit(pointers['expr'][1])
        node.val = pointers['expr'][1].val + pointers['term'][0].val

    def expr_1(self, node):
        pointers = self.__helper__(node)
        pointers['term'][0].env = node.env
        self.visit(pointers['term'][0])
        pointers['expr'][1].env = node.env
        self.visit(pointers['expr'][1])
        node.val = pointers['expr'][1].val - pointers['term'][0].val

    def expr_2(self, node):
        pointers = self.__helper__(node)
        pointers['term'][0].env = node.env
        self.visit(pointers['term'][0])
        node.val = pointers['term'][0].val

    def term_0(self, node):
        pointers = self.__helper__(node)
        pointers['term'][1].env = node.env
        self.visit(pointers['term'][1])
        pointers['factor'][0].env = node.env
        self.visit(pointers['factor'][0])
        node.val = pointers['term'][1].val * pointers['factor'][0].val

    def term_1(self, node):
        pointers = self.__helper__(node)
        pointers['term'][1].env = node.env
        self.visit(pointers['term'][1])
        pointers['factor'][0].env = node.env
        self.visit(pointers['factor'][0])
        node.val = pointers['term'][1].val / pointers['factor'][0].val

    def term_2(self, node):
        pointers = self.__helper__(node)
        pointers['factor'][0].env = node.env
        self.visit(pointers['factor'][0])
        node.val = pointers['factor'][0].val

    def factor_0(self, node):
        pointers = self.__helper__(node)
        pointers['factor'][1].env = node.env
        self.visit(pointers['factor'][1])
        node.val = - pointers['factor'][1].val

    def factor_1(self, node):
        pointers = self.__helper__(node)
        pointers['literal'][0].env = node.env
        self.visit(pointers['literal'][0])
        node.val = pointers['literal'][0].val

    def literal_0(self, node):
        pointers = self.__helper__(node)
        node.val = node.env.get(str(pointers['ID'][0]),0)
        if(not(str(pointers['ID'][0]) in node.env)):
            raise Exception("Unbound variable")

    def literal_1(self, node):
        pointers = self.__helper__(node)
        node.val = float(pointers['NUMBER'][0])

    def literal_2(self, node):
        pointers = self.__helper__(node)
        pointers['expr'][0].env = node.env
        self.visit(pointers['expr'][0])
        node.val = pointers['expr'][0].val

In [ ]:
parser = Lark(grammar)
tree = parser.parse("let x = 20; let y = x*2; x + -y")
MyInterpreter().visit(tree)
print(tree.val)

-20.0


In [ ]:
parser = Lark(grammar)
tree = parser.parse("let x = 20; let y = x*2; x + -y + z")
MyInterpreter().visit(tree)
print(tree.val)

Exception: Unbound variable

# Exercise 1

- Consider a list of values with mixed types, whose sum of the numeric values must be positive

- E.g., `( um , 1, a, dois, b, c, -1,  dd, -4, erro, 12, 4 )` is valid, but `( um , 1,a, dois, b, c, -1,  dd, -4, erro,  4 )` is invalid


## Exercise 1.1

- Write an AG for finding the **maximum** value of a mixed type list. Write the CFG, the attributes, their evaluation rules, and context conditions.

In [ ]:
grammar = r"""
SA(start) { avg : float }
SA(elems) { num_count : int, num_sum : int }
SA(elem) { value:int }

start : "(" elems ")"
    ER {
        start.avg = elems.num_sum / elems.num_count
    }
    CC{
        elems.num_sum <= 0 -> "Non positive sum";
    }

elems : elems "," elem
    ER{
        elems[1].num_count = elems[2].num_count + (1 if elem.value else 0);
        elems[1].num_sum = elems[2].num_sum + (elem.value or 0);
    }

      | elem
      ER {
        elems.num_count = 1 if elem.value else 0; 
        elems.num_sum = elem.value or 0;
      }

elem : NUMBER
    ER {
        elem.value = int(NUMBER);
    }

    | TXT
    ER {
        elem.value = NONE;
    }

NUMBER : /[0-9]+/
TXT : /[a-zA-Z]+/
"""

## Exercise 1.2

- Implement this AG using `LarkAG`

In [ ]:
import lark_ag
def mixed_lists(lst):
    """
    >>> mixed_lists("( 1, 2, 3, 2, 1 )")
    1.8
    >>> mixed_lists("( 1, -2, -3, 2, 1 )")
    Traceback (most recent call last):
        ...
    Exception: Non-positive sum
    >>> mixed_lists("( comentario, -2, nota, alto, 5, baixo, 4, 12, 4 )")
    4.6
    >>> mixed_lists("( comentario, -2, nota, alto, 5, baixo, 4, -12, 4 )")
    Traceback (most recent call last):
        ...
    Exception: Non-positive sum
    """

    parser = lark_ag.Lark_AG(grammar)
    res = parser.process(lst)
    return res.avg


import doctest
doctest.run_docstring_examples(mixed_lists, globals())

## Exercise 1.3

- Consider now that there are two special keywords that can occur in the list: `p`, after which all numerical values must be positive, and `n` after which all numerical values must be negative

- Rewrite the AG and its implementation in `LarkAG`, still for calculating the maximum value, but checking the validity of the numerical values

In [ ]:
grammar = r"""

start : "(" elems ")"
ER {
    start.avg = elems.num_sum / elems.num_count; 
}

CC {
    elems.num_sum < 0 -> "Soma negativa";
}

elems : elem
ER {
    elems.num_count = 1 if elem.val else 0; 
    elems.num_sum = elem.val or 0;
    elem.sinalH = None;
    elems.ctx = elem.sinal; 
}

CC {
    !elem.ok -> "Valores trocados";
}

| elems "," elem
ER {
    elems[1].num_count = elems[2].num_count + (1 if elem.val else 0); 
    elems[1].num_sum = elemes[2].num_sum + (elem.val or 0);
    elem.sinalH = elems[2].ctx;
    elems[1].ctx = elem.sinal;
}

CC {
    !elem.ok -> "Valores trocados";
}

elem : NUM
ER {
    elem.val = int(NUM);
    elem.ok = True if (elem.sinal and elem.val > 0) or (!elem.sinal and elem.val < 0) else False;
    elem.sinal = elem.sinalH;
}

| NEG
ER {
    elem.val = None;
    elem.sinal = False;
    elem.ok = True;
}

| POS
ER {
    elem.val = None;
    elem.sinal = True;
    elem.ok = True;
}
    
NUM : /.?\d+/
NEG : "n";
POS : "p";
"""

import lark_ag

def mixed_lists(lst):
    """
    >>> mixed_lists("( p, 1, 2, 3, 2, 1 )")
    1.8
    >>> mixed_lists("( n, -1, -2, -5, -4, p, n, p, 1, 30, 2 )")
    3.0
    >>> mixed_lists("( p, 1, -2, -3, -4, p, 1, 30, 2 )")  x 
    Traceback (most recent call last):
        ...
    Exception: Wrong signal element
    >>> mixed_lists("( n, -1, -200, -3, -4, p, 1, 30, 2 )")
    Traceback (most recent call last):
        ...
    Exception: Non-positive sum
    """

    parser = lark_ag.Lark_AG(grammar)
    res = parser.process(lst)
    return res.avg

import doctest
doctest.run_docstring_examples(mixed_lists, globals())

## Exercise 1.4

- We now want to implement contextual addition for these lists: whatever numerical values appear between special keyword `start` and `stop` is processed, anything else is ignored; a list may contain multiple `start` and `stop` segments

- Adapt the AG to represet this contextual addition, and implement it in `LarkAG`

In [ ]:
grammar = r"""
SA(start) { avg : float }
SA(elems) { num_count : int, num_sum : int, ctx : str }
SA(elem) { val : int, ok : bool }
IA(elem) { ok_h : bool }

start : "(" elems ")"
ER	{
    start.avg = elems.num_sum / elems.num_count;
}
CC {
    elems.num_sum > 0 -> "Non-positive sum";
}

elems : elem
ER {
    elems.num_count = 0;
    elems.num_sum = 0;
    elem.ok_h = False;
    elems.ctx = elem.ok;
}

| elems "," elem
ER {
    elems[1].num_count = elems[2].num_count + (1 if elem.val else 0);
    elems[1].num_sum = elems[2].num_sum + (elem.val or 0);
    elem.ok_h = elems[2].ctx;
    elems[1].ctx = elem.ok;
}


elem : NUM
ER {
    elem.val = int(NUM);
    elem.ok = elem.ok_h;
}

| START
ER {
    elem.val = 0;
    elem.ok = True;
}

| STOP
ER {
    elem.val = 0;
    elem.ok = False;
}

NUM : /-?\d+/
START : "start"
STOP : "stop"
"""

import lark_ag

def mixed_lists(lst):
    """
    >>> mixed_lists("( start, 1, 2, 3, 2, 1, stop )")
    1.8
    >>> mixed_lists("( -6, -8, start, 1, 2, 3, 2, 1, stop, -8, -9 )")
    1.8
    >>> mixed_lists("( -6, -8, start, 1, 2, stop, 2, 6, start, 3, 2, 1, stop, -8, -9 )")
    1.8
    >>> mixed_lists("( start, -9, 2, -3, 2, 1, stop )")
    Traceback (most recent call last):
        ...
    Exception: Non-positive sum
    """

    parser = lark_ag.Lark_AG(grammar)
    res = parser.process(lst)
    return res.avg

import doctest
doctest.run_docstring_examples(mixed_lists, globals())

**********************************************************************
File "__main__", line 58, in NoName
Failed example:
    mixed_lists("( -6, -8, start, 1, 2, 3, 2, 1, stop, -8, -9 )")
Exception raised:
    Traceback (most recent call last):
      File "/home/guilherme/anaconda3/envs/EG/lib/python3.10/doctest.py", line 1350, in __run
        exec(compile(example.source, filename, "single",
      File "<doctest NoName[1]>", line 1, in <module>
        mixed_lists("( -6, -8, start, 1, 2, 3, 2, 1, stop, -8, -9 )")
      File "/tmp/ipykernel_18899/950841814.py", line 69, in mixed_lists
        res = parser.process(lst)
      File "/home/guilherme/anaconda3/envs/EG/lib/python3.10/site-packages/lark_ag/lark_ag.py", line 12, in process
        return self.processor.process(input)
      File "/home/guilherme/anaconda3/envs/EG/lib/python3.10/site-packages/lark_ag/processor_layer/processor_layer.py", line 10, in process
        self.interpreter.visit(tree)
      File "/home/guilherme/.local/

# Exercise 2

Let's revisit the language for first-order logic formulas with the following syntax:

- Predicates $A$, $B$, $C$, ...
- Variables $a$, $b$, $c$, ...
- Constants false $⊥$ and true $⊤$
- Predicate application $A(a_1, \ldots, a_n)$ if $A$ has arity $n$
- Conjunction $f_1 ∧ f_2$,
- Disjunction $f_1 ∨ f_2$,
- Negation $¬ f_1$,
- Implication $f_1 → f_2$
- Equivalence $f_1 ↔︎ f_2$
- Universal quantification $∀ a . f_1$
- Existential quantification $∃ a . f_1$

## Exercise 2.1

- Write an AG for finding all occurrences of predicates within a formula, validating whether predicate and variable ocurrences are valid. Write the CFG, the attributes, their evaluation rules, and context conditions.

## Exercise 2.2

- Implement this AG using `LarkAG`

# Exercise 3

Consider a language to represent nested lists of strings, such as
```
("Introduction", "Background" ("Formalism", "Tools"), "Approach" ("Overview", "Implementation" ("Architecture", "Evaluation"), "Discussion"), "Conclusion")
```

## Exercise 3.1

Write an AG that translates such lists into a numbered format with indentation, such as the following for the previous example.
```
1. Introduction
2. Background
  2.1. Formalism
  2.2. Tools
3. Approach
  3.1 Overview
  3.2 Implementation
     3.1.1 Architecture
     3.1.2 Evaluation
  3.3 Discussion
4. Conclusion
```

In [1]:
grammar = r"""
SA(elem) { output : str }
IA(elem) { prefix : int, index : int }
SA(elems) { output : str, quantity : int }
IA(elems) { prefix : int }
SA(lista) { output : str }
IA(lista) { prefix : int }

start : lista
ER {
    lista.prefix = [];
}
TR {
    print(lista.output);
}

lista : "(" elems ")"
ER {
    lista.output = elems.output;
    elems.prefix = lista.prefix;
}

elems : elems "," elem
ER {
    elems[1].output = elems[2].output + "\n" + elem.output;
    elems[2].prefix = elems[1].prefix;
    elem.prefix = elems[1].prefix;
    elem.index = elems[2].quantity + 1;
    elems[1].quantity = elem.index;
}
| elem
ER {
    elems.output = elem.output;
    elem.prefix = elems.prefix;
    elem.index = 1;
    elems.quantity = elem.index;
}

elem : STR
ER {
    elem.output = f"{len(elem.prefix)*'\t'}{".".join(map(str,elem.prefix+[elem.index]))}. {STR}";
}
| STR lista
ER {
    elem.output = f"{len(elem.prefix)*'\t'}{".".join(map(str,elem.prefix+[elem.index]))}. {STR}\n{lista.output}";
    lista.prefix = elem.prefix + [elem.index];
}

STR : /[A-Za-z"]+/
"""

In [ ]:
grammar = r"""
SA(familia) { pares : list, num_filhos : int }
SA(apelido) { txt : str }
SA(filhos) { pares : list, num_filhos : int }
SA(nomes) { pares : list, num_filhos : int }
IA(nomes) { apel:str }

start : familias
familias : familia

| familias ";" familia
familia : pai "+" mae ":" filhos
ER { familia.pares = filhos.pares; 
     familia.num_filhos = filhos.num_filhos; 
    }

CC { 
    familia.num_filhos < 2 -> "Não é permitido familia com menos do que dois filhos"; 
}

pai : NOME
mae : NOME

filhos : nomes "-" apelido
ER { nomes.apel = apelido.txt;
    filhos.pares = nomes.pares; 
    filhos.num_filhos = nomes.num_filhos;
    }

nomes : NOME
ER {
nomes.pares = [(str(NOME),nomes.apel)]; 
nomes.num_filhos = 1;
}

| nomes "," NOME
ER {
nomes[1].pares = nomes[2].pares+[(str(NOME),nomes[1].apel)];
nomes[2].apel = nomes[1].apel; 
nomes[1].num_filhos = nomes[2].num_filhos + 1}

apelido : NOME
ER { apelido.txt = str(NOME); }

NOME : /[a-zA-Z]+/
"""

## Exercise 3.2

Implement this AG using `LarkAG`.

In [2]:
import lark_ag

parser = lark_ag.Lark_AG(grammar)
parser.process('(Introduction, Background (Formalism, Tools), Approach (Overview, Implementation (Architecture, Evaluation), Discussion), Conclusion)')

SyntaxError: f-string expression part cannot include a backslash (<string>, line 50)

-- Nuno Macedo

In [ ]:
from lark import Lark, Visitor

grammar = """
start: employees_block schedule_block

employees_block: "employees" "{" employee_list "}"
employee_list: employee+
employee: "employee" ID ":" STRING "preferences" "{" preferences "}"
preferences: availability_stmt preferred_shift_stmt
availability_stmt: "availability:" date_list ";"
date_list: DATE ("," DATE)*
preferred_shift_stmt: "preferredShift:" SHIFT ";"

schedule_block: "schedule" "{" assign_list "}"
assign_list: assign_stmt+
assign_stmt: "assign" ID "to" SHIFT "on" DATE ";"

ID: /[a-zA-Z_][a-zA-Z0-9_]*/
STRING: /"[^"]*"/
DATE: /\d{4}-\d{2}-\d{2}/
SHIFT: "morning" | "evening"

%import common.WS
%ignore WS
"""

# Visitor para validar o schedule
class ScheduleValidator(Visitor):
    def __init__(self):
        self.employees = {}  # {ID: {"name": STRING, "availability": [...], "shift": SHIFT}}
        self.errors = []

    def employee(self, tree):
        id_token = str(tree.children[0])
        name_token = str(tree.children[1])[1:-1]  # remove aspas
        availability_node = tree.children[2].children[0]  # availability_stmt
        dates = [str(date) for date in availability_node.children[0].children]  # date_list
        shift = str(tree.children[2].children[1].children[0])  # preferredShift
        self.employees[id_token] = {"name": name_token, "availability": dates, "shift": shift}

    def assign_stmt(self, tree):
        id_token = str(tree.children[0])
        shift_token = str(tree.children[1])
        date_token = str(tree.children[2])Engenharia

        # 1️⃣ Verifica se o funcionário existe
        if id_token not in self.employees:
            self.errors.append(f"Error: Employee '{id_token}' assigned but not defined.")
            return

        emp = self.employees[id_token]

        # 2️⃣ Verifica se a data está disponível
        if date_token not in emp["availability"]:
            self.errors.append(f"Error: Employee '{id_token}' not available on {date_token}.")

        # 3️⃣ Verifica se o shift coincide com o preferredShift
        if shift_token != emp["shift"]:
            self.errors.append(f"Error: Employee '{id_token}' assigned to {shift_token} but prefers {emp['shift']}.")

# Exemplo de uso
parser = Lark(grammar, parser="lalr")
text = """
employees {
employee alice: "Alice Johnson" preferences {
availability: 2025-03-21, 2025-03-22, 2025-03-23;
preferredShift: morning;
}
employee bob: "Bob Smith" preferences {
availability: 2025-03-24, 2025-03-25;
preferredShift: evening;
} }
schedule {
assign alice to morning on 2025-03-21;
assign bob to evening on 2025-03-24;
}
"""

tree = parser.parse(text)
validator = ScheduleValidator()
validator.visit(tree)

if validator.errors:
    print("Schedule is INVALID:")
    for e in validator.errors:
        print(e)
else:
    print("Schedule is VALID")


Schedule is VALID
